# 🐝 BeeWatch — Audio Anomaly Detection dengan Autoencoder

Notebook ini melatih **Audio Autoencoder** sebagai komponen deteksi anomali sistem BeeWatch.
Model belajar pola akustik normal koloni lebah — reconstruction error tinggi = kondisi menyimpang = anomali.

Dibangun menggunakan **TensorFlow + Keras** sesuai arsitektur sistem BeeWatch.

## Cara Pakai
1. Di Kaggle: klik **+ Add Data** → cari `annajyang/beehive-sounds` → tambahkan
2. Attach dataset cache: **+ Add Data** → cari `beewatch-cache` → tambahkan
3. **Run All** dari atas ke bawah
4. Download output: `audio_autoencoder.keras`, `audio_scaler.pkl`, `model_config.pkl`, `thresholds.pkl`
5. Copy ke folder server Flask BeeWatch

## Pendekatan
- **Audio Autoencoder**: belajar pola akustik normal koloni lebah dari 7100 file WAV.
  Rekaman 1 menit setiap 15 menit dari node ESP32 audio (INMP441).
- Data CSV dipakai hanya untuk referensi timeline (tidak digunakan untuk training model).
- Pendekatan unsupervised murni — tidak membutuhkan label, cocok untuk deployment nyata di lapangan.


## 📦 1. Setup dan Import

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
warnings.filterwarnings('ignore')

import librosa
import librosa.display

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# GPU check
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow version : {tf.__version__}')
print(f'GPU tersedia       : {len(gpus)} — {[g.name for g in gpus] if gpus else "CPU mode"}')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)


2026-06-07 06:46:07.743398: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780814768.284233      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780814768.397555      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780814769.747058      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780814769.747100      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780814769.747103      58 computation_placer.cc:177] computation placer alr

TensorFlow version : 2.19.0
GPU tersedia       : 2 — ['/physical_device:GPU:0', '/physical_device:GPU:1']


## 📂 2. Load Dataset

**Pastikan dataset `annajyang/beehive-sounds` sudah di-attach ke notebook ini.**

Dataset berisi:
- File `.wav` — rekaman suara sarang dari 4 koloni lebah madu Eropa
- `all_data_updated.csv` — data lingkungan + label queen presence

In [2]:
BASE_DIR  = '/kaggle/input/datasets/annajyang/beehive-sounds'
AUDIO_DIR = os.path.join(BASE_DIR, 'sound_files', 'sound_files')
CSV_PATH  = os.path.join(BASE_DIR, 'all_data_updated.csv')

df_meta = pd.read_csv(CSV_PATH)
print(f'Total baris CSV : {len(df_meta)}')
print(f'Kolom           : {list(df_meta.columns)}')
print(f'\nDistribusi queen presence:')
print(df_meta['queen presence'].value_counts())
df_meta.head()

Total baris CSV : 1275
Kolom           : ['device', 'hive number', 'date', 'hive temp', 'hive humidity', 'hive pressure', 'weather temp', 'weather humidity', 'weather pressure', 'wind speed', 'gust speed', 'weatherID', 'cloud coverage', 'rain', 'lat', 'long', 'file name', 'queen presence', 'queen acceptance', 'frames', 'target', 'time', 'queen status']

Distribusi queen presence:
queen presence
1    1117
0     158
Name: count, dtype: int64


,device,hive number,date,hive temp,hive humidity,hive pressure,weather temp,weather humidity,weather pressure,wind speed,...,rain,lat,long,file name,queen presence,queen acceptance,frames,target,time,queen status
0,1,5,2022-06-08 14:52:28,36.42,30.29,1007.45,26.68,52,1013,8.75,...,0,37.29,-121.95,2022-06-08--14-52-28_1.raw,1,2,8,0,0.583,0
1,1,5,2022-06-08 15:51:41,33.56,33.98,1006.93,25.99,53,1012,10.29,...,0,37.29,-121.95,2022-06-08--15-51-41_1.raw,1,2,8,0,0.625,0
2,1,5,2022-06-08 17:21:53,29.01,42.73,1006.68,24.49,56,1012,8.75,...,0,37.29,-121.95,2022-06-08--17-21-53_1.raw,0,0,8,1,0.708,1
3,1,5,2022-06-08 18:20:59,30.51,36.74,1006.68,22.97,59,1012,8.23,...,0,37.29,-121.95,2022-06-08--18-20-59_1.raw,0,0,8,1,0.750,1
4,1,5,2022-06-08 19:20:04,30.32,35.55,1006.58,21.52,61,1012,7.20,...,0,37.29,-121.95,2022-06-08--19-20-04_1.raw,0,0,8,1,0.792,1


In [3]:
audio_files = []
for root, dirs, files in os.walk(AUDIO_DIR):
    for f in sorted(files):
        if f.lower().endswith('.wav'):
            audio_files.append(os.path.join(root, f))

print(f'Total file WAV ditemukan: {len(audio_files)}')
if audio_files:
    print(f'Contoh: {audio_files[0]}')

Total file WAV ditemukan: 7100
Contoh: /kaggle/input/datasets/annajyang/beehive-sounds/sound_files/sound_files/2022-06-05--17-41-01_2__segment0.wav


## 🎧 3. Ekstraksi Fitur Audio

Fitur per file WAV (total 72 dimensi):
- MFCC 13 koefisien × mean+std = 26
- Mel-Spectrogram 20 band × mean+std = 40
- Spectral Centroid mean+std = 2
- Spectral Rolloff mean+std = 2
- Zero Crossing Rate mean+std = 2

Setiap file dipotong ke segmen **2 detik**, fitur per segmen dirata-rata.

In [4]:
SR               = 16000
SEGMENT_DURATION = 2.0
N_MFCC           = 13
N_MELS           = 20
N_FFT            = 512
HOP_LENGTH       = 256

# Cache path: cek dari uploaded dataset dulu, baru fallback ke working dir
# Cara pakai: upload audio_features_cache.pkl sebagai Kaggle dataset
# bernama "beewatch-cache", lalu attach ke notebook ini via Add Data
CACHE_PATH_INPUT  = '/kaggle/input/datasets/kaytarechia/beewatch-cache/audio_features_cache.pkl'
CACHE_PATH_OUTPUT = '/kaggle/working/audio_features_cache.pkl'

if os.path.exists(CACHE_PATH_INPUT):
    CACHE_PATH = CACHE_PATH_INPUT
    print(f'Cache ditemukan di input dataset: {CACHE_PATH_INPUT}')
else:
    CACHE_PATH = CACHE_PATH_OUTPUT
    print(f'Cache input tidak ditemukan, akan simpan ke: {CACHE_PATH_OUTPUT}')

def extract_audio_features(file_path, sr=SR, n_mfcc=N_MFCC):
    try:
        y, _ = librosa.load(file_path, sr=sr, mono=True)
    except Exception as e:
        return None
    seg_len = int(SEGMENT_DURATION * sr)
    n_segs  = len(y) // seg_len
    if n_segs == 0:
        return None
    seg_feats = []
    for i in range(n_segs):
        seg    = y[i*seg_len:(i+1)*seg_len]
        mfcc   = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=n_mfcc, n_fft=N_FFT, hop_length=HOP_LENGTH)
        mel    = librosa.feature.melspectrogram(y=seg, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=128)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        sc     = librosa.feature.spectral_centroid(y=seg, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        srf    = librosa.feature.spectral_rolloff(y=seg, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        zcr    = librosa.feature.zero_crossing_rate(y=seg, hop_length=HOP_LENGTH)
        feat   = np.concatenate([
            np.mean(mfcc, axis=1), np.std(mfcc, axis=1),
            np.mean(mel_db, axis=1)[:N_MELS], np.std(mel_db, axis=1)[:N_MELS],
            [np.mean(sc), np.std(sc)],
            [np.mean(srf), np.std(srf)],
            [np.mean(zcr), np.std(zcr)]
        ])
        seg_feats.append(feat)
    return np.mean(seg_feats, axis=0)

# Test fungsi
if audio_files:
    t = extract_audio_features(audio_files[0])
    if t is not None:
        print(f'Dimensi fitur: {t.shape[0]}')  # harus 72

# Load cache atau ekstraksi ulang
if os.path.exists(CACHE_PATH):
    audio_features_list = joblib.load(CACHE_PATH)
    audio_df = pd.DataFrame(audio_features_list)
    print(f'Load dari cache! Total: {len(audio_df)} file')
else:
    print(f'Cache tidak ditemukan. Memproses {len(audio_files)} file WAV...')
    print(f'(Setelah selesai, download output/audio_features_cache.pkl dan upload sebagai dataset "beewatch-cache")')
    audio_features_list = []
    for i, wav_path in enumerate(audio_files):
        feat = extract_audio_features(wav_path)
        if feat is not None:
            audio_features_list.append({
                'filename'  : os.path.basename(wav_path),
                'hive_num'  : 1,
                'timestamp' : pd.Timestamp('2024-01-01') + pd.Timedelta(minutes=15*i),
                'features'  : feat,
            })
        if (i+1) % 20 == 0:
            print(f'  {i+1}/{len(audio_files)} selesai...')
    audio_df = pd.DataFrame(audio_features_list)
    # Selalu simpan ke working dir agar bisa didownload
    joblib.dump(audio_features_list, CACHE_PATH_OUTPUT)
    print(f'Selesai! Total: {len(audio_df)} file')
    print(f'Cache disimpan ke {CACHE_PATH_OUTPUT} — download dan upload sebagai dataset "beewatch-cache"')


Cache input tidak ditemukan, akan simpan ke: /kaggle/working/audio_features_cache.pkl
Dimensi fitur: 72
Cache tidak ditemukan. Memproses 7100 file WAV...
(Setelah selesai, download output/audio_features_cache.pkl dan upload sebagai dataset "beewatch-cache")
  20/7100 selesai...
  40/7100 selesai...
  60/7100 selesai...


KeyboardInterrupt: 

## 🌡️ 4. Load Metadata CSV

CSV `all_data_updated.csv` berisi metadata koloni lebah.
Dipakai hanya untuk membuat referensi timeline (1271 interval 15 menit),
**tidak digunakan untuk training model** — model dilatih secara unsupervised dari fitur WAV.


In [ ]:
# Load CSV hanya untuk referensi timeline
df_meta_clean = df_meta[['hive temp']].dropna().copy()
df_meta_clean['timestamp'] = pd.date_range(
    start='2024-01-01', periods=len(df_meta_clean), freq='15min'
)
df_timeline = df_meta_clean[['timestamp']].reset_index(drop=True)

print(f'Total sampel CSV (referensi timeline) : {len(df_timeline)}')
print('CSV dimuat — hanya untuk timeline, tidak digunakan untuk training.')


In [ ]:
# Tidak ada visualisasi distribusi sensor (bukan bagian dari model audio)
print('Lanjut ke normalisasi fitur audio...')


## 🔧 5. Normalisasi

StandardScaler diterapkan pada fitur audio sebelum training.
`audio_scaler.pkl` **wajib disimpan** — harus dipakai ulang saat inferensi di server.


In [ ]:
audio_feature_matrix     = np.vstack(audio_df['features'].values)
audio_scaler             = StandardScaler()
audio_feature_normalized = audio_scaler.fit_transform(audio_feature_matrix)

joblib.dump(audio_scaler, 'audio_scaler.pkl')

print(f'Audio normalized shape : {audio_feature_normalized.shape}')
print('audio_scaler.pkl disimpan!')


In [ ]:
# Visualisasi PCA audio features
pca_vis  = PCA(n_components=2)
audio_2d = pca_vis.fit_transform(audio_feature_normalized)

plt.figure(figsize=(9, 5))
plt.scatter(audio_2d[:, 0], audio_2d[:, 1], alpha=0.4, s=15, color='steelblue')
plt.xlabel(f'PC1 ({pca_vis.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca_vis.explained_variance_ratio_[1]:.2%})')
plt.title('PCA Audio Features (2D)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
print(f'Total audio files : {len(audio_df)}')
print(f'Audio feature dim : {audio_feature_normalized.shape[1]}')
print()
print('Pendekatan: Unsupervised Anomaly Detection')
print('  - Model dilatih dengan semua data audio (semua dianggap representasi normal)')
print('  - Reconstruction error tinggi saat inferensi = kondisi akustik menyimpang = anomali')
print('  - Tidak membutuhkan label — sesuai kondisi deployment nyata di lapangan')


## 🧠 6. Definisi Model: Audio Autoencoder

Sistem BeeWatch menggunakan **Audio Autoencoder** dengan **Keras Functional API**.

### Arsitektur (v3)
Progressive compression: **72 → 64 → 32** (bottleneck) → 64 → 72.
Aktivasi: **ReLU** + **BatchNormalization** + **Dropout(0.1)**

**Prinsip deteksi:** Ketika kondisi akustik sarang menyimpang dari pola normal yang dipelajari,
reconstruction error tinggi → **anomali terdeteksi → notifikasi Telegram dikirim**.

Model disimpan format **`.keras`** — mudah di-load untuk inferensi di server Flask.


In [ ]:
def build_audio_autoencoder(input_dim, encoding_dim=32):
    """
    Audio Autoencoder — Keras Functional API.

    Arsitektur (v3 Audio-Only):
        Encoder : input_dim → 64 → 32 (bottleneck)
        Decoder : 32 → 64 → input_dim
    """
    inp = keras.Input(shape=(input_dim,), name='audio_input')
    x = layers.Dense(64, name='enc_dense1')(inp)
    x = layers.BatchNormalization(name='enc_bn1')(x)
    x = layers.Activation('relu', name='enc_relu1')(x)
    x = layers.Dropout(0.1, name='enc_drop1')(x)
    latent = layers.Dense(encoding_dim, activation='relu', name='latent')(x)
    x = layers.Dense(64, name='dec_dense1')(latent)
    x = layers.BatchNormalization(name='dec_bn1')(x)
    x = layers.Activation('relu', name='dec_relu1')(x)
    x = layers.Dropout(0.1, name='dec_drop1')(x)
    output = layers.Dense(input_dim, name='audio_output')(x)
    autoencoder = Model(inp, output, name='AudioAutoencoder')
    encoder     = Model(inp, latent, name='AudioEncoder')
    return autoencoder, encoder


audio_input_dim = audio_feature_normalized.shape[1]  # 72
audio_autoencoder, audio_encoder = build_audio_autoencoder(audio_input_dim, encoding_dim=32)
audio_autoencoder.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse')
print(audio_autoencoder.summary())


In [ ]:
print('=' * 58)
print('AUDIO AUTOENCODER — Komponen Utama BeeWatch')
print('=' * 58)
print(f'  Input     : {audio_input_dim} dimensi (MFCC + Mel-Spectrogram + Spectral)')
print(f'  Encoder   : {audio_input_dim} → 64 → 32 (bottleneck)')
print(f'  Decoder   : 32 → 64 → {audio_input_dim}')
print(f'  Aktivasi  : ReLU + BatchNormalization + Dropout(0.1)')
print(f'  Parameter : {audio_autoencoder.count_params():,}')


## 🏋️ 7. Training

In [ ]:
from sklearn.model_selection import train_test_split

X_a_train, X_a_val = train_test_split(audio_feature_normalized, test_size=0.20, random_state=42)
print(f'Audio train/val : {len(X_a_train)} / {len(X_a_val)} (total {len(audio_feature_normalized)})')

# FIX: best_epoch sebelumnya 148/150 → sama seperti masalah sensor v1
# Naikkan ke 250 agar EarlyStopping yang memutuskan kapan berhenti
audio_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=20,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=8, min_lr=1e-6, verbose=1)
]

print('\nMulai training Audio Autoencoder...')
audio_history = audio_autoencoder.fit(
    X_a_train, X_a_train,
    validation_data=(X_a_val, X_a_val),
    epochs=250,          # FIX: naik dari 150 (sebelumnya best_epoch=148/150 belum konvergen)
    batch_size=32,
    callbacks=audio_callbacks,
    verbose=0
)

best_epoch_a     = np.argmin(audio_history.history['val_loss'])
audio_losses_tr  = audio_history.history['loss']
audio_losses_val = audio_history.history['val_loss']
print(f'  Audio — best epoch: {best_epoch_a+1}, best val loss: {min(audio_losses_val):.6f}')
print('Training selesai!')


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(audio_losses_tr,  color='steelblue', linewidth=2, label='Train')
plt.plot(audio_losses_val, color='tomato', linewidth=2, linestyle='--', label='Val')
plt.axvline(best_epoch_a, color='gray', linestyle=':', alpha=0.7, label=f'Best epoch {best_epoch_a+1}')
plt.title('Audio Autoencoder — Training Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()


## 🚨 8. Deteksi Anomali

In [ ]:
def compute_reconstruction_error(model, data):
    recon = model.predict(data, batch_size=64, verbose=0)
    return np.mean((data - recon) ** 2, axis=1)

def normalize_errors(e):
    return (e - e.min()) / (e.max() - e.min() + 1e-8)


# Hitung reconstruction error (semua 7100 sampel audio)
audio_errors      = compute_reconstruction_error(audio_autoencoder, audio_feature_normalized)
audio_errors_norm = normalize_errors(audio_errors)
print(f'Audio errors — Mean: {audio_errors.mean():.6f}, Std: {audio_errors.std():.6f}')

# Align ke panjang timeline (1271 sampel)
n_tl                 = len(df_timeline)
idx_audio            = np.linspace(0, len(audio_errors_norm) - 1, n_tl, dtype=int)
audio_errors_aligned = audio_errors_norm[idx_audio]
print(f'Audio errors aligned : {len(audio_errors_aligned)} sampel')

# Results DataFrame
results_df = df_timeline.copy().reset_index(drop=True)
results_df['audio_score'] = audio_errors_aligned

# Threshold persentil 95
threshold = np.percentile(results_df['audio_score'], 95)
results_df['is_anomaly'] = results_df['audio_score'] > threshold

print(f'\nThreshold P95 : {threshold:.4f}')
print(f'Anomali       : {results_df["is_anomaly"].sum()} dari {len(results_df)} ({results_df["is_anomaly"].mean()*100:.1f}%)')


## 📈 9. Visualisasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(results_df['audio_score'], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold P95 ({threshold:.3f})')
axes[0].set_xlabel('Audio Anomaly Score (normalized)')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Anomaly Score')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

scores = results_df['audio_score'].values
anom_mask = results_df['is_anomaly'].values
axes[1].plot(scores, alpha=0.6, linewidth=0.8, color='steelblue', label='Audio Score')
axes[1].axhline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.3f})')
pred_idx = np.where(anom_mask)[0]
if len(pred_idx):
    axes[1].scatter(pred_idx, scores[pred_idx], color='tomato', s=20, zorder=5, label='Terdeteksi Anomali')
axes[1].set_xlabel('Indeks Sampel (15 menit interval)')
axes[1].set_ylabel('Anomaly Score')
axes[1].set_title('Anomaly Score Over Time')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('anomaly_scores.png', dpi=150)
plt.show()


## 📊 10. Evaluasi Model

Evaluasi dilakukan dengan pendekatan **unsupervised** — tanpa label eksternal.
Fokus utama pada distribusi reconstruction error Audio Autoencoder sebagai komponen utama sistem.


In [ ]:
print('=' * 55)
print('EVALUASI MODEL — UNSUPERVISED ANOMALY DETECTION')
print('=' * 55)
print('\nAudio Reconstruction Error (normalized, 7100 sampel):')
print(f'  Mean  : {audio_errors_norm.mean():.4f}')
print(f'  Std   : {audio_errors_norm.std():.4f}')
print(f'  Min   : {audio_errors_norm.min():.4f}')
print(f'  Max   : {audio_errors_norm.max():.4f}')
print(f'  P95   : {np.percentile(audio_errors_norm, 95):.4f}')
print('\nAnomaly Score (aligned ke 1271 sampel timeline):')
print(f'  Mean      : {results_df["audio_score"].mean():.4f}')
print(f'  Std       : {results_df["audio_score"].std():.4f}')
print(f'  Threshold : {threshold:.4f} (persentil 95)')
print(f'  Anomali   : {results_df["is_anomaly"].sum()} ({results_df["is_anomaly"].mean()*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(audio_errors_norm, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(np.percentile(audio_errors_norm, 95), color='red', linestyle='--',
                label=f'P95 = {np.percentile(audio_errors_norm, 95):.3f}')
axes[0].set_title('Distribusi Audio Reconstruction Error (7100 sampel)')
axes[0].set_xlabel('Error (normalized)'); axes[0].set_ylabel('Frekuensi')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].hist(results_df['audio_score'], bins=50, color='mediumpurple', alpha=0.7, edgecolor='black')
axes[1].axvline(threshold, color='red', linestyle='--', label=f'Threshold = {threshold:.3f}')
axes[1].set_title('Distribusi Anomaly Score (1271 sampel timeline)')
axes[1].set_xlabel('Score'); axes[1].set_ylabel('Frekuensi')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('error_distributions.png', dpi=150)
plt.show()


In [ ]:
print('Analisis Sensitivitas Threshold:')
print(f'{"Persentil":>10} {"Threshold":>12} {"Anomali (n)":>12} {"Anomali (%)":>12}')
print('-' * 50)
for pct in [90, 92, 95, 97, 98, 99]:
    thr_val  = np.percentile(results_df['audio_score'], pct)
    n_anom   = int((results_df['audio_score'] > thr_val).sum())
    marker   = ' <-- dipilih' if pct == 95 else ''
    print(f'{pct:>10} {thr_val:>12.4f} {n_anom:>12} {n_anom/len(results_df)*100:>11.1f}%{marker}')
print(f'\nThreshold aktif : {threshold:.4f} (persentil 95)')
print(f'Sampel anomali  : {results_df["is_anomaly"].sum()} dari {len(results_df)}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot([audio_errors_norm], labels=['Audio AE'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='steelblue'),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Reconstruction Error Distribution')
axes[0].set_ylabel('Error (normalized)')
axes[0].grid(True, alpha=0.3)

normal_pts  = results_df[~results_df['is_anomaly']]
anomaly_pts = results_df[results_df['is_anomaly']]
axes[1].scatter(normal_pts.index, normal_pts['audio_score'],
                c='steelblue', alpha=0.3, s=10, label='Normal')
axes[1].scatter(anomaly_pts.index, anomaly_pts['audio_score'],
                c='tomato', alpha=0.8, s=40, marker='x', label='Anomali')
axes[1].axhline(threshold, color='red', linestyle='--', linewidth=1.5,
                label=f'Threshold ({threshold:.3f})')
axes[1].set_xlabel('Indeks Sampel'); axes[1].set_ylabel('Audio Anomaly Score')
axes[1].set_title('Normal vs Terdeteksi Anomali')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150)
plt.show()


In [ ]:
from sklearn.decomposition import PCA

idx_vis    = np.linspace(0, len(audio_feature_normalized) - 1, len(df_timeline), dtype=int)
latent_vis = audio_encoder.predict(audio_feature_normalized[idx_vis], batch_size=64, verbose=0)

pca_a = PCA(n_components=2)
a_2d  = pca_a.fit_transform(latent_vis)

plt.figure(figsize=(9, 6))
sc = plt.scatter(a_2d[:, 0], a_2d[:, 1],
                 c=results_df['audio_score'].values, cmap='YlOrRd', alpha=0.6, s=15)
plt.colorbar(sc, label='Audio Anomaly Score')
plt.title('Audio Latent Space (PCA 2D) — warna = Anomaly Score')
plt.xlabel(f'PC1 ({pca_a.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca_a.explained_variance_ratio_[1]:.1%})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('latent_space.png', dpi=150)
plt.show()

print(f'Audio PCA: PC1={pca_a.explained_variance_ratio_[0]:.1%}, '
      f'PC2={pca_a.explained_variance_ratio_[1]:.1%}, '
      f'Total={sum(pca_a.explained_variance_ratio_):.1%}')


## 💾 11. Simpan Model dan Artefak

Download file berikut dari panel **Output** Kaggle → copy ke folder server Flask BeeWatch.

| File | Fungsi |
|---|---|
| `audio_autoencoder.keras` | Model untuk inferensi di server |
| `audio_scaler.pkl` | StandardScaler — wajib ada di server |
| `model_config.pkl` | Konfigurasi ekstraksi fitur |
| `thresholds.pkl` | Threshold anomali P95 |


In [ ]:
print('=' * 55)
print(f"{'RINGKASAN HASIL TRAINING & EVALUASI':^55}")
print('=' * 55)
print(f"{'Metrik':<35} {'Nilai':>18}")
print('-' * 55)
print(f"{'Training samples':<35} {len(X_a_train):>18,}")
print(f"{'Validation samples':<35} {len(X_a_val):>18,}")
print(f"{'Epochs (best val)':<35} {best_epoch_a+1:>18}")
print(f"{'Best val loss':<35} {min(audio_losses_val):>18.4f}")
print(f"{'Reconstruction error mean':<35} {audio_errors_norm.mean():>18.4f}")
print(f"{'Reconstruction error std':<35} {audio_errors_norm.std():>18.4f}")
print(f"{'Threshold P95':<35} {threshold:>18.4f}")
print(f"{'Anomali terdeteksi':<35} {results_df['is_anomaly'].sum():>14} (5.0%)")
print('=' * 55)


In [ ]:
audio_autoencoder.save('audio_autoencoder.keras')
audio_encoder.save('audio_encoder.keras')

config = {
    'audio_input_dim' : audio_input_dim,
    'audio_encoding'  : 32,
    'audio_arch'      : [64, 32],
    'audio_dropout'   : 0.1,
    'sr'              : SR,
    'segment_duration': SEGMENT_DURATION,
    'n_mfcc'          : N_MFCC,
    'n_mels'          : N_MELS,
    'n_fft'           : N_FFT,
    'hop_length'      : HOP_LENGTH,
    'version'         : 'v3_audio_only',
}
joblib.dump(config, 'model_config.pkl')

thresholds = {
    'audio_threshold_pct95': float(threshold),
    'audio_mean'           : float(audio_errors_norm.mean()),
    'audio_std'            : float(audio_errors_norm.std()),
    # Legacy keys untuk kompatibilitas server Flask
    'combined_threshold'   : float(threshold),
    'w_audio'              : 1.0,
    'w_sensor'             : 0.0,
}
joblib.dump(thresholds, 'thresholds.pkl')

print('FILE OUTPUT (audio only):')
for f in ['audio_autoencoder.keras', 'audio_encoder.keras',
          'audio_scaler.pkl', 'model_config.pkl', 'thresholds.pkl']:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f)/1024:.1f} KB' if exists else 'tidak ada'
    print(f'  {"✅" if exists else "❌"} {f:<35} {size}')
print('\nSemua artefak audio tersimpan!')


In [ ]:
import tensorflow as tf
import json

# Load dari config + weights
with open('config.json') as f:
    cfg = json.load(f)

model = tf.keras.models.model_from_json(json.dumps(cfg))
model.load_weights('model_weights.h5')

# Simpan ke format .keras
model.save('audio_autoencoder.keras')
print("Done — audio_autoencoder.keras siap")

## 🔌 12. Kode Inferensi untuk Server Flask

Copy fungsi-fungsi ini ke `server.py` — dijalankan setiap kali ESP32 Audio mengirim file WAV.


In [ ]:
inference_snippet = '''
# ─────────────────────────────────────────────────────────────────────
# BeeWatch — Audio Inference Module untuk Server Flask
#
# ALUR:
#   1. ESP32 Audio kirim WAV via HTTP POST ke /upload/audio
#   2. Server simpan ke /tmp/, panggil analyze_audio()
#   3. librosa ekstrak 72 fitur → audio_scaler → audio model
#   4. Hitung reconstruction error → score → severity
#   5. Kalau is_anomaly=True → kirim notifikasi Telegram
#
# INSTALL: pip install tensorflow librosa joblib scikit-learn numpy flask
# ─────────────────────────────────────────────────────────────────────

import numpy as np, librosa, joblib
from tensorflow import keras

cfg      = joblib.load("model_config.pkl")
audio_sc = joblib.load("audio_scaler.pkl")
thr      = joblib.load("thresholds.pkl")
audio_m  = keras.models.load_model("audio_autoencoder.keras")
_ = audio_m.predict(np.zeros((1, cfg["audio_input_dim"])), verbose=0)
print("Audio model loaded.")


def _extract_features(seg, sr, cfg):
    n_mfcc = cfg["n_mfcc"]; n_mels = cfg["n_mels"]
    n_fft  = cfg["n_fft"];  hop    = cfg["hop_length"]
    mfcc   = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop)
    mel    = librosa.feature.melspectrogram(y=seg, sr=sr, n_fft=n_fft, hop_length=hop, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    sc     = librosa.feature.spectral_centroid(y=seg, sr=sr, n_fft=n_fft, hop_length=hop)
    srf    = librosa.feature.spectral_rolloff(y=seg, sr=sr, n_fft=n_fft, hop_length=hop)
    zcr    = librosa.feature.zero_crossing_rate(y=seg, hop_length=hop)
    return np.concatenate([
        np.mean(mfcc, axis=1), np.std(mfcc, axis=1),
        np.mean(mel_db, axis=1)[:n_mels], np.std(mel_db, axis=1)[:n_mels],
        [np.mean(sc), np.std(sc), np.mean(srf), np.std(srf), np.mean(zcr), np.std(zcr)]
    ])


def analyze_audio(wav_path):
    sr = cfg["sr"]; sl = int(cfg["segment_duration"] * sr)
    try: audio, _ = librosa.load(wav_path, sr=sr, mono=True)
    except Exception as e:
        return {"audio_score": 0.0, "is_anomaly": False, "severity": "normal", "error": str(e)}

    n_segs = len(audio) // sl
    if n_segs == 0:
        return {"audio_score": 0.0, "is_anomaly": False, "severity": "normal", "n_segments": 0}

    feats       = [_extract_features(audio[i*sl:(i+1)*sl], sr, cfg) for i in range(n_segs)]
    feat_scaled = audio_sc.transform(np.mean(feats, axis=0).reshape(1, -1))
    recon       = audio_m.predict(feat_scaled, verbose=0)
    raw_err     = float(np.mean((feat_scaled - recon) ** 2))

    # Normalisasi ke 0-1: mean+3*std = batas atas
    upper = thr["audio_mean"] + 3 * thr["audio_std"]
    score = float(np.clip((raw_err - thr["audio_mean"]) / (upper - thr["audio_mean"] + 1e-8), 0, 1))
    thresh = thr["audio_threshold_pct95"]

    return {
        "audio_score" : score,
        "is_anomaly"  : score > thresh,
        "severity"    : "critical" if score > thresh*1.5 else ("warning" if score > thresh else "normal"),
        "raw_error"   : raw_err,
        "n_segments"  : n_segs
    }


# ── Flask endpoint ────────────────────────────────────────────────────
# @app.route("/upload/audio", methods=["POST"])
# def upload_audio():
#     wav_path = "/tmp/rec.wav"
#     request.files["audio"].save(wav_path)
#     result = analyze_audio(wav_path)
#     if result["is_anomaly"]:
#         send_telegram_alert(result["severity"])
#     return jsonify(result)
'''

print('Kode inferensi untuk server.py siap di-copy.')
print(inference_snippet)


## ✅ 13. Ringkasan

### Perubahan dari v1
| # | Isu | Sebelum | Sesudah |
|---|---|---|---|
| 1 | Arsitektur encoder | 72→**128**→64→32 (expand dulu) | 72→64→32 (progressive compress) |
| 2 | Dropout | `Dropout(0.2)` | `Dropout(0.1)` |
| 3 | Max epochs | 150 (best epoch 148 — belum konvergen) | 250 (EarlyStopping yang putuskan) |
| 4 | Scope | Dual AE (audio + sensor) | **Audio-only** |

### File Output
| File | Fungsi |
|---|---|
| `audio_autoencoder.keras` | Model utama untuk inferensi di server |
| `audio_scaler.pkl` | StandardScaler — wajib ada, dipakai saat inferensi |
| `model_config.pkl` | Konfigurasi ekstraksi fitur (SR, N_MFCC, dll.) |
| `thresholds.pkl` | Threshold anomali P95 |

### Alur di Server
```
ESP32 Audio → kirim WAV → Flask /upload/audio
           → librosa ekstrak 72 fitur akustik
           → audio_scaler.transform()
           → audio_autoencoder.keras → reconstruction error
           → score > threshold? → Telegram notification
```

### Catatan untuk Laporan
- Framework: **TensorFlow + Keras** — model disimpan format `.keras`
- Input: 72 fitur akustik per file WAV (MFCC × 13, Mel-Spectrogram × 20, Spectral × 6)
- Pendekatan unsupervised — tidak membutuhkan label data training
- Threshold ditetapkan pada persentil 95 reconstruction error (5% error tertinggi = anomali)
